# 1. Environment Setup

In [ ]:
# Local environment setup (replaces Databricks %run ./.setup/learner_setup)
import os
from dotenv import load_dotenv

load_dotenv()  # Loads variables from .env file in the project root

# Environment variables from learner_setup (telemetry opt-out, tiktoken cache)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
tiktoken_cache_dir = os.path.abspath("./.setup/tiktoken_cache/")
os.environ["TIKTOKEN_CACHE_DIR"] = tiktoken_cache_dir
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

## 1.2 Load Necessary Libraries

In [ ]:
import json
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain_core.tools import tool
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage, HumanMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from IPython.display import display, Image, Markdown
from langgraph.checkpoint.memory import MemorySaver
import os
from dotenv import load_dotenv
import httpx
import pandas as pd

## 1.3 Configure Azure OpenAI

In [ ]:
def get_access_token():
    # OAuth2 token endpoint for UHG API
    auth = "https://api.uhg.com/oauth2/token"
    # Scope for the API
    scope = "https://api.uhg.com/.default"
    # Grant type for client credentials flow
    grant_type = "client_credentials"

    with httpx.Client() as client:
        # Prepare request body with credentials from environment variables
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": os.environ["AZURE_CLIENT_ID"],
            "client_secret": os.environ["AZURE_CLIENT_SECRET"],
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        # Request access token from UHG API
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        # Extract access token from response
        access_token = resp.json()["access_token"]
        return access_token

# Azure OpenAI API endpoint
AZURE_OPENAI_ENDPOINT = "https://api.uhg.com/api/cloud/api-management/ai-gateway/1.0"
# Name of the deployed chat model
MODEL_DEPLOYMENT_NAME = "gpt-4o-mini_2024-07-18"
# OpenAI API version
OPENAI_API_VERSION = "2023-05-15"
# Project ID for API requests
PROJECT_ID = "2db8b475-ec3e-468b-b253-058bbe8ff77b"
# Name of the deployed embeddings model
EMBEDDINGS_DEPLOYMENT_NAME = "text-embedding-ada-002_2"

def get_chat_client():
    # Returns an AzureChatOpenAI client configured for the specified deployment
    return AzureChatOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_version=OPENAI_API_VERSION,
        azure_deployment=MODEL_DEPLOYMENT_NAME,
        temperature=0,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": PROJECT_ID
        }
    )

def get_embeddings_client():
    # Returns an AzureOpenAIEmbeddings client configured for the specified deployment
    return AzureOpenAIEmbeddings(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_version=OPENAI_API_VERSION,
        azure_deployment=EMBEDDINGS_DEPLOYMENT_NAME,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": PROJECT_ID
        }
    )

# 2. Data Loading and Preprocessing

## 2.1 Load and Inspect Validation Patient Data

In [ ]:
# Loading pre-built datasets for web search and PubMed
with open('./Data/validation_records.json') as f:
    sample_patients = json.load(f)

## 2.2 Implement Utility Function for Computing Age

In [ ]:
from datetime import datetime

def compute_age(date_of_birth: str, reference_date: str) -> int:
    dob = datetime.strptime(date_of_birth, "%Y-%m-%d")
    ref = datetime.strptime(reference_date, "%Y-%m-%d")
    age = ref.year - dob.year - ((ref.month, ref.day) < (dob.month, dob.day))
    return age

for record in sample_patients:
    record['age'] = compute_age(record['date_of_birth'], record['date_of_service'])

## 2.3 Load and Inspect Insurance Policy Data

In [ ]:
with open('./Data/insurance_policies.json') as f:
    insurance_policies = json.load(f)

## 2.4 Load and Inspect Reference Code Data Files

In [ ]:
with open('./Data/reference_codes.json') as f:
    ref = json.load(f)
    CPT_CODES = ref['CPT']
    ICD10_CODES = ref['ICD10']

# 3. Create and Define Tools for the ReAct Agent

In [ ]:
# Creating a get patient by id function

all_patients = sample_patients.copy()

def get_patient_by_id(patient_id):
    """
    Retrieve patient info from sample_patients by patient_id.
    Returns the patient info dict if found, otherwise returns None.
    """
    return next(
        (record for record in all_patients if record.get('patient_id') == patient_id),
        None
    )

# Create a dictionary-like interface for easy access
class PatientDB:
    def __getitem__(self, patient_id):
        return get_patient_by_id(patient_id)

PATIENT_DB = PatientDB()

# Creating a get policy by id function

def get_policy_by_id(policy_id):
    """
    Retrieve policy info from insurance_policies by policy_id.
    Returns the policy info dict if found, otherwise returns None.
    """
    return next(
        (policy for policy in insurance_policies if policy.get('policy_id') == policy_id),
        None
    )

class PolicyDB:
    def __getitem__(self, policy_id):
        return get_policy_by_id(policy_id)

POLICY_DB = PolicyDB()


## 3.1 Define Tool for Summarizing Patient Records

In [ ]:
@tool
def summarize_patient_record(patient_id: str) -> str:
    """
    Summarizes a patient record and associated insurance claim data using LLM-powered reasoning.
    Takes a patient_id and retrieves the full patient record from the database.
    Returns a structured summary string for eligibility reasoning including ALL relevant details.
    """
    # Retrieve the patient record by patient_id
    patient = get_patient_by_id(patient_id)
    if not patient:
        # Return message if patient not found
        return f"No patient found with patient_id: {patient_id}"
    
    # Compose the prompt for the LLM to generate a structured summary of the patient record
    prompt = f"""
You are a medical eligibility assistant. Summarize the following patient record and insurance claim data. 
Provide a clear, structured summary with these sections: 
1. Patient Demographics (patient_id, age, gender, date_of_birth, etc.)
2. Insurance Policy ID
3. Diagnoses and Descriptions (with ICD-10 codes)
4. Procedures and Descriptions (with CPT codes)
5. Preauthorization Status (required and obtained - include boolean values)
6. Billed Amount
7. Date of Service

CRITICAL: Include ALL fields from the patient record, especially:
- Age (calculated from date_of_birth)
- Gender (male/female) - THIS IS MANDATORY
- Any authorization flags
- All procedure and diagnosis codes

Patient Record and Claim Data:
{json.dumps(patient, indent=2)}

Structured Summary:
"""
    # Get the chat client (LLM interface)
    chat_client = get_chat_client()
    # Invoke the LLM with the constructed prompt and get the response
    response = chat_client.invoke([HumanMessage(content=prompt)])
    # Return the LLM's response content (patient summary)
    return response.content

In [ ]:
PATIENT_DB['P011']
patient_summary = summarize_patient_record('P011')
display(Markdown(patient_summary))

## 3.2 Define Tool for Summarizing Insurance Policy Guideline

In [ ]:
@tool
def summarize_insurance_policy(policy_id: str) -> str:
    """
    Uses LLM reasoning to summarize the claim coverage rules and conditions of an insurance policy,
    ensuring the claim coverage process follows the defined criteria.
    Returns a structured summary string with ALL coverage criteria including age, gender, and other restrictions.
    """
    # Retrieve the insurance policy by policy_id
    policy = next((p for p in insurance_policies if p.get('policy_id') == policy_id), None)
    if not policy:
        return f"No insurance policy found for policy_id: {policy_id}"

    # Compose the prompt for the LLM to generate a structured summary of the policy
    prompt = f"""
You are an expert insurance policy assistant. Summarize the claim coverage rules and conditions 
of the following insurance policy. Ensure your summary clearly explains:

1. Covered services and procedures (list specific CPT codes if provided)
2. Covered diagnoses (list specific ICD-10 codes if provided)

3. **ELIGIBILITY RESTRICTIONS** - This is CRITICAL - YOU MUST CHECK EVERY FIELD FOR THESE RESTRICTIONS:

   IMPORTANT: Look in ALL fields including 'covered_services', 'eligibility_criteria', 'covered_procedures', 
   'covered_diagnoses', and any nested fields for restriction information!

   a) **AGE RESTRICTIONS**:
      - Check 'eligibility_criteria' field and any procedure-specific criteria
      - State the minimum and maximum age requirements explicitly
      - Format: 'Age requirement: X to Y years old'
      - If found in eligibility_criteria, state: "Age restriction found: [exact text]"

   b) **GENDER RESTRICTIONS** (CRITICALLY IMPORTANT):
      - Check EVERY procedure in 'covered_services' or 'covered_procedures' for gender restrictions
      - Look in 'eligibility_criteria' for gender-related text like "male only", "female only", "males", "females", "men", "women"
      - Check each CPT code's description and conditions for gender restrictions
      - If you find ANY gender restriction for ANY procedure, you MUST state it explicitly:
        * "CPT [code] - [procedure name]: Gender requirement: MALE ONLY" OR
        * "CPT [code] - [procedure name]: Gender requirement: FEMALE ONLY" OR
        * "CPT [code] - [procedure name]: Gender requirement: NO RESTRICTION"
      - If eligibility_criteria mentions gender, quote it exactly: "Gender restriction found: [exact text from policy]"
      - DEFAULT: If no gender info found for a procedure, state "Gender requirement: NOT SPECIFIED (assume no restriction)"

   c) **GEOGRAPHIC RESTRICTIONS**:
      - Check 'eligibility_criteria' and policy fields for location/state/region restrictions
      - Any state or region limitations

   d) **MEMBERSHIP/PLAN TYPE RESTRICTIONS**:
      - Check 'eligibility_criteria' for plan type requirements
      - Any specific plan or membership requirements

   e) **OTHER ELIGIBILITY CRITERIA**:
      - Any other restrictions mentioned in 'eligibility_criteria' field or elsewhere
      - Look for keywords: "only", "must", "required", "restricted", "limited to"

4. Exclusions and limitations

5. Preauthorization requirements - BE VERY SPECIFIC:
   - List which procedures/services REQUIRE preauthorization
   - List which procedures/services DO NOT require preauthorization
   - If no preauth requirements are specified, state 'No preauthorization requirements specified'

6. Claim submission process

7. Any special conditions or notes

CRITICAL INSTRUCTION FOR GENDER:
- You MUST examine the 'eligibility_criteria' field word by word for gender-related terms
- You MUST check each covered procedure for gender-specific language
- If the word "male", "female", "men", or "women" appears ANYWHERE in the policy, you MUST report it
- When in doubt, quote the exact text from the policy that mentions gender

Insurance Policy Data:
{json.dumps(policy, indent=2)}

Structured Summary (MUST include explicit gender requirements for each procedure):
"""
    # Get the chat client (LLM interface)
    chat_client = get_chat_client()
    # Invoke the LLM with the constructed prompt and get the response
    response = chat_client.invoke([HumanMessage(content=prompt)])
    # Return the LLM's response content (policy summary)
    return response.content

In [ ]:
POLICY_DB['POL1002']

In [ ]:
policy_summary = summarize_insurance_policy.invoke('POL1002')
display(Markdown(policy_summary))

## 3.3 Define Tool for Claim Coverage Check

In [ ]:
@tool
def claim_coverage_check(patient_record_summary: str, policy_summary: str) -> str:
    """
    Tool that checks whether a patient's claimed procedures are covered by their insurance policy.
    Returns a step-by-step coverage analysis and final decision (APPROVE or ROUTE FOR REVIEW).
    MUST evaluate ALL criteria including age, gender, geographic, and other eligibility restrictions, 
    procedure coverage, diagnosis appropriateness, and preauthorization.
    """
    # Compose the prompt for the LLM, instructing it to follow a detailed, step-by-step coverage analysis
    prompt = f"""
You are a medical insurance eligibility analyst. Using the structured summaries below, 
assess whether each claimed procedure in the patient's record satisfies the coverage conditions 
outlined in their insurance policy.

CRITICAL EVALUATION STEPS for each procedure:

Step 1: Is the procedure code explicitly covered in the policy?

Step 2: Is the diagnosis code appropriate and covered for this procedure?

Step 3: **ELIGIBILITY RESTRICTIONS CHECK** (CRITICALLY IMPORTANT - Check ALL of these):

   a) **AGE ELIGIBILITY**:
      - What is the patient's age?
      - What are the policy's age requirements?
      - Does the patient's age fall within the allowed range?
      - If patient age is OUTSIDE the policy range → ROUTE FOR REVIEW

   b) **GENDER ELIGIBILITY** (MUST CHECK CAREFULLY):
      - What is the patient's gender (male or female)?
      - Does the policy summary mention ANY gender restrictions for this specific procedure?
      - Look for phrases like:
        * "male only", "males only", "men only"
        * "female only", "females only", "women only"
        * "Gender requirement: MALE ONLY"
        * "Gender requirement: FEMALE ONLY"
      - CRITICAL LOGIC:
        * If policy says "male only" AND patient is female → ROUTE FOR REVIEW (explain gender mismatch)
        * If policy says "female only" AND patient is male → ROUTE FOR REVIEW (explain gender mismatch)
        * If policy says "no restriction" OR "not specified" → Gender requirement SATISFIED
        * If unclear, look at the exact policy text quoted in the summary
      - You MUST explicitly state: "Policy requires [male/female/no restriction], patient is [male/female]"
      - If there's a mismatch, clearly state it prevents approval

   c) **GEOGRAPHIC ELIGIBILITY**:
      - Does the patient's location meet policy requirements?
      - If geographic restrictions exist and not met → ROUTE FOR REVIEW

   d) **PLAN/MEMBERSHIP ELIGIBILITY**:
      - Does the patient's plan type meet requirements?
      - If plan restrictions exist and not met → ROUTE FOR REVIEW

   e) **OTHER ELIGIBILITY CRITERIA**:
      - Check for any other restrictions in the policy
      - If any other criteria not met → ROUTE FOR REVIEW

Step 4: **PREAUTHORIZATION CHECK**:
   a) Does the policy state this procedure REQUIRES preauthorization?
   b) If YES to (a): Is preauthorization obtained? If not obtained → ROUTE FOR REVIEW
   c) If NO to (a) OR policy doesn't mention preauth for this procedure → Preauth requirement is SATISFIED

Step 5: Are there any other exclusions that apply?

DECISION LOGIC:
✓ APPROVE if ALL of the following are true:
  - Procedure is explicitly covered
  - Diagnosis is appropriate and covered
  - **Patient age is within policy's allowed age range**
  - **Patient gender matches policy requirements (if policy specifies male/female only)**
  - **Patient location meets policy requirements (if any geographic restrictions exist)**
  - **Patient plan type meets policy requirements (if any plan restrictions exist)**
  - **All other eligibility criteria are satisfied**
  - Preauth requirement is satisfied (either obtained OR not required)
  - No other exclusions apply

✗ ROUTE FOR REVIEW if ANY of the following are true:
  - Procedure not covered
  - Diagnosis not appropriate
  - **Patient age is outside policy's allowed age range**
  - **Patient gender doesn't match policy gender requirements (e.g., policy requires male but patient is female)**
  - **Patient location doesn't meet geographic restrictions**
  - **Patient plan type doesn't meet policy requirements**
  - **Any other eligibility criteria not met**
  - Preauth required but not obtained
  - Other exclusions apply
  - Information is unclear or ambiguous

IMPORTANT: Do NOT say "no specific gender restrictions" if the policy summary explicitly states a gender requirement.
Read the policy summary carefully and quote any gender restrictions you find.

Patient Record Summary:
{patient_record_summary}

Insurance Policy Summary:
{policy_summary}

Provide your analysis following these steps exactly, then state your final decision with detailed reasoning.
IMPORTANT: If routing for review due to any eligibility restriction, explicitly state:
- The specific restriction (age/gender/geographic/plan/other)
- The policy requirement (quote exact text if available)
- The patient's actual characteristic
- Why this prevents approval

Step-by-step Coverage Analysis:
"""

    # Get the chat client (LLM interface)
    chat_client = get_chat_client()
    # Invoke the LLM with the constructed prompt and get the response
    response = chat_client.invoke([HumanMessage(content=prompt)])
    # Return the LLM's response content (coverage analysis and decision)
    return response.content

In [ ]:
result = claim_coverage_check.invoke(
    {
        'patient_record_summary': patient_summary,
        'policy_summary': policy_summary
    }
)

display(Markdown(result))

In [ ]:
# Creating the tools list
tools = [
    summarize_patient_record,
    summarize_insurance_policy,
    claim_coverage_check
]

# 4. System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are an Insurance Claims Reviewer Agent. Your job is to evaluate insurance claims by using the provided tools in sequence.

WORKFLOW - Follow these steps IN ORDER:

1. **Extract Information from User Query**: When the user provides a patient record, extract the patient_id and policy_id from it.

2. **Call summarize_patient_record**: Pass ONLY the patient_id (e.g., 'P011', 'P012') to get a complete structured summary of the patient's claim including:
   - Patient demographics (especially AGE and GENDER)
   - Diagnoses with ICD-10 codes
   - Procedures with CPT codes
   - Preauthorization status
   - Billed amount
   - Geographic location (if available)
   - Plan/membership type (if available)

3. **Call summarize_insurance_policy**: Pass the policy_id from the patient record to get the policy rules including:
   - Covered procedures and diagnoses
   - **ALL ELIGIBILITY RESTRICTIONS**:
     * AGE restrictions (min/max age)
     * GENDER restrictions (male only/female only/no restriction)
     * GEOGRAPHIC restrictions (state/region limitations)
     * PLAN/MEMBERSHIP restrictions
     * OTHER eligibility criteria
   - Preauthorization requirements
   - Exclusions and limitations

4. **Call claim_coverage_check**: Pass both summaries to get a detailed coverage analysis that checks:
   - Procedure coverage
   - Diagnosis appropriateness
   - **ALL ELIGIBILITY RESTRICTIONS**:
     * AGE ELIGIBILITY (patient age must be within policy's age range)
     * GENDER ELIGIBILITY (patient gender must match policy requirements)
     * GEOGRAPHIC ELIGIBILITY (patient location must meet policy requirements)
     * PLAN/MEMBERSHIP ELIGIBILITY (patient plan must meet policy requirements)
     * OTHER ELIGIBILITY CRITERIA
   - Preauthorization requirements
   - Other exclusions

CRITICAL RULES:
- Always check if patient meets ALL eligibility restrictions (age, gender, geographic, plan, etc.)
- If patient AGE is outside policy's allowed age range → MUST route for review
- If patient GENDER doesn't match policy's gender requirement → MUST route for review
- If patient LOCATION doesn't meet geographic restrictions → MUST route for review
- If patient PLAN doesn't meet plan requirements → MUST route for review
- Only APPROVE when ALL criteria are satisfied (procedure covered, diagnosis appropriate, ALL eligibility restrictions met, preauth satisfied, no exclusions)
- ROUTE FOR REVIEW when ANY criterion is not met or unclear

Final output format:
Decision: [APPROVE or ROUTE FOR REVIEW]
Reason: [Write a single, concise, flowing paragraph (NOT bullet points) that mentions the specific procedure codes, diagnosis codes, and explains which eligibility factors were checked and why the decision was made. For denials, clearly state which specific requirement was not met.]
"""

# 5. Create LangGraph Claims Approval ReAct Agent

In [ ]:
# Create the system prompt for the ReAct agent using the defined SYSTEM_PROMPT
REACT_AGENT_PROMPT = SystemMessage(content=SYSTEM_PROMPT)

# Define the agent's state structure for LangGraph
class State(TypedDict):
    messages: Annotated[list, add_messages]  # Conversation history
    patient_summary: str                     # Patient summary from tool
    policy_summary: str                      # Policy summary from tool
    coverage_result: str                     # Coverage check result

# Node: Calls the LLM with the current state and available tools
def tool_calling_llm(state: State) -> State:
    # Trim messages to fit within token limits for the LLM
    trimmed_state = trim_messages(
        state["messages"],
        max_tokens=127000,
        strategy="last",
        token_counter=count_tokens_approximately,
        include_system=True,
        allow_partial=True
    )
    # Prepend system prompt to the conversation
    state_with_instructions = [REACT_AGENT_PROMPT] + trimmed_state
    llm = get_chat_client()
    llm_with_tools = llm.bind_tools(tools)
    # Invoke the LLM with tools and get the response
    response = llm_with_tools.invoke(state_with_instructions)
    return {
        "messages": state["messages"] + [response],
        "patient_summary": state.get("patient_summary", ""),
        "policy_summary": state.get("policy_summary", ""),
        "coverage_result": state.get("coverage_result", "")
    }

# Condition function: Determines if the next step is to call tools or end
def tools_condition(state: State):
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "END"

# Build the LangGraph state machine for the ReAct agent
react_builder = StateGraph(State)
react_builder.add_node("tool_calling_llm", tool_calling_llm)  # LLM node
react_builder.add_node("tools", ToolNode(tools=tools))        # Tools node
react_builder.add_edge(START, "tool_calling_llm")             # Start edge
react_builder.add_conditional_edges(
    "tool_calling_llm",
    tools_condition,
    {"tools": "tools", "END": END}
)
react_builder.add_edge("tools", "tool_calling_llm")           # Loop back after tool use

# Set up memory for the agent and compile the agent
memory = MemorySaver()
insurance_claim_agent = react_builder.compile(checkpointer=memory)

# 6. Define the Utility Function to Call the Agent

In [ ]:
def call_react_agent(agent, prompt, user_session_id, verbose=False):
    """
    Utility function to call the react agent and stream its step-by-step reasoning for a specific user.
    Streams the agent's execution using the given prompt and session ID for memory tracking.
    Includes error handling to report issues and allow the notebook to continue.
    """
    try:
        # Start streaming events from the agent using the provided prompt and session ID
        events = agent.stream(
            {"messages": [{"role": "user", "content": prompt}]},
            {"configurable": {"thread_id": user_session_id}},
            stream_mode="values"
        )

        print('Running Agent. Please wait...')

        # Iterate through streamed events and optionally pretty print intermediate messages
        for event in events:
            if verbose and "messages" in event and event["messages"]:
                last_msg = event["messages"][-1]
                if hasattr(last_msg, "pretty_print"):
                    last_msg.pretty_print()

        # Display the agent's final response using Markdown rendering
        print('\n\nAgent Final Response:\n')
        if "messages" in event and event["messages"]:
            display(Markdown(event["messages"][-1].content))
    except Exception as e:
        # Handle and print any errors that occur during agent execution
        print("\n[ERROR] Agent call failed:", str(e))
        import traceback
        traceback.print_exc()
        print("\nThe notebook is still running. You may proceed to the next case or investigate the error above.")

# 7. Run the Agent on Validation Patient Data

In [ ]:
call_react_agent(
    agent=insurance_claim_agent,
    prompt='Evaluate this claim: "P016"',
    user_session_id="example-session-001",
    verbose=True
)

## 7.2 Compare Results to Human Reference Data

In [ ]:
# Run agent on each patient and collect results
validation_results = []
for i, record in enumerate(sample_patients):
    patient_id = record['patient_id']
    session_id = f"test-session-{patient_id}"
    
    print(f"Processing {i+1}/{len(sample_patients)}: {patient_id}")
    
    try:
        response = get_agent_response(insurance_claim_agent, patient_id, session_id)
        validation_results.append({
            'patient_id': patient_id,
            'generated_response': response
        })
    except Exception as e:
        print(f"Error processing {patient_id}: {e}")
        validation_results.append({
            'patient_id': patient_id,
            'generated_response': f"Error: {str(e)}"
        })

print(f"\nProcessed {len(validation_results)} records")  # Print summary

# Load reference results
reference_df = pd.read_csv('./Data/validation_reference_results.csv')

# Create DataFrame from generated results
generated_df = pd.DataFrame(validation_results)

# Merge on patient_id for comparison
comparison_df = reference_df.merge(generated_df, on='patient_id', how='left', suffixes=('_reference', '_generated'))

display(comparison_df)

In [ ]:
# Show all content in each cell without truncation
display(comparison_df.style.set_properties(**{'white-space': 'pre-wrap'}))

# 8. Test Dataset Evaluation

Load test records and process them through the agent to generate submission results.

## 8.1 Load Test Records

In [ ]:
# Load test records from test_records.json
with open('./Data/test_records.json') as f:
    test_records = json.load(f)

# Compute age for each test record
for record in test_records:
    record['age'] = compute_age(record['date_of_birth'], record['date_of_service'])

all_patients = sample_patients + test_records

## 8.2 Process Test Records Through Agent

In [ ]:
# Function to get agent response without printing
def get_agent_response(agent, patient_id, session_id):
    """
    Get the agent's final response for a given patient ID without verbose output.
    Returns only the final decision text.
    """
    prompt = f'Evaluate this claim: "{patient_id}"'
    
    events = agent.stream(
        {"messages": [{"role": "user", "content": prompt}]},
        {"configurable": {"thread_id": session_id}},
        stream_mode="values"
    )
    
    # Get the final event
    final_event = None
    for event in events:
        final_event = event
    
    # Extract final response
    if final_event and "messages" in final_event and final_event["messages"]:
        return final_event["messages"][-1].content
    return "No response generated"

# Process all test records
results = []
print("Processing test records through the agent...")

for i, record in enumerate(test_records):
    patient_id = record['patient_id']
    session_id = f"test-session-{patient_id}"
    
    print(f"Processing {i+1}/{len(test_records)}: {patient_id}")
    
    try:
        response = get_agent_response(insurance_claim_agent, patient_id, session_id)
        results.append({
            'patient_id': patient_id,
            'generated_response': response
        })
    except Exception as e:
        print(f"Error processing {patient_id}: {e}")
        results.append({
            'patient_id': patient_id,
            'generated_response': f"Error: {str(e)}"
        })

print(f"\nProcessed {len(results)} records")

## 8.3 Save Results to submission.csv

In [ ]:
# Create DataFrame and save to CSV
submission_df = pd.DataFrame(results)

# Save to submission.csv
submission_df.to_csv('submission.csv', index=False)

df = pd.read_csv("submission.csv")
from IPython.display import Markdown
display(Markdown(df.to_markdown(index=False)))